### Load the cleaned FIRMS data

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    r"C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML"
)

CLEANED_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed_data"
    / "firms_cleaned.csv"
)

df = pd.read_csv(CLEANED_FILE)

print("Shape:", df.shape)
print(df.head())

Shape: (702968, 12)
   latitude  longitude    acq_date  acq_time      satellite  brightness  \
0   8.98492   77.60124  2025-01-01       801  NOAA-20 VIIRS      341.70   
1   9.00014   77.61005  2025-01-01       801  NOAA-20 VIIRS      330.68   
2   9.27809   78.04165  2025-01-01       801  NOAA-20 VIIRS      330.24   
3   9.56098   77.63283  2025-01-01      1949  NOAA-21 VIIRS      303.48   
4   9.56323   77.63605  2025-01-01      1949  NOAA-21 VIIRS      303.55   

  confidence   frp daynight version  detections_same_location_date  \
0          n  8.35        D       2                              1   
1          n  2.68        D       2                              1   
2          n  1.58        D       2                              1   
3          n  0.98        N  2.0NRT                              1   
4          n  1.23        N  2.0NRT                              1   

                 source_file  
0  DL_FIRE_J1V-C2_812115.zip  
1  DL_FIRE_J1V-C2_812115.zip  
2  DL_FIRE_J1V-

### Create timestamp

In [2]:
df["acq_date"] = pd.to_datetime(
    df["acq_date"],
    errors="coerce"
)

time_str = (
    pd.to_numeric(
        df["acq_time"],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
    .astype(str)
    .str.zfill(4)
)

df["timestamp"] = pd.to_datetime(
    df["acq_date"].dt.strftime("%Y-%m-%d")
    + " "
    + time_str,
    format="%Y-%m-%d %H%M",
    errors="coerce"
)

df = df.dropna(
    subset=[
        "latitude",
        "longitude",
        "timestamp",
        "frp"
    ]
).copy()

df = df.sort_values("timestamp").reset_index(drop=True)

print("Final shape:", df.shape)
print("Time range:")
print(df["timestamp"].min())
print(df["timestamp"].max())

Final shape: (702968, 13)
Time range:
2025-01-01 03:50:00
2025-03-19 21:31:00


### Create spatial cells

In [3]:
SPATIAL_CELL_KM = 1.0

LAT_CELL = SPATIAL_CELL_KM / 111.0
LON_CELL = SPATIAL_CELL_KM / 100.0

df["lat_cell"] = (
    np.floor(df["latitude"] / LAT_CELL)
    .astype("int32")
)

df["lon_cell"] = (
    np.floor(df["longitude"] / LON_CELL)
    .astype("int32")
)

print(
    "Unique spatial cells:",
    df[["lat_cell", "lon_cell"]]
    .drop_duplicates()
    .shape[0]
)

Unique spatial cells: 257899


### Historical detection frequency

In [4]:
df["date_only"] = df["timestamp"].dt.date

historical_count = (
    df.groupby(
        ["lat_cell", "lon_cell"]
    )
    .size()
    .rename("historical_detection_count")
    .reset_index()
)

df = df.merge(
    historical_count,
    on=["lat_cell", "lon_cell"],
    how="left"
)

print(
    df["historical_detection_count"]
    .describe()
)

count    702968.000000
mean         46.373425
std         152.019311
min           1.000000
25%           2.000000
50%           4.000000
75%           9.000000
max        1572.000000
Name: historical_detection_count, dtype: float64


### Historical FRP behaviour

In [5]:
historical_frp = (
    df.groupby(
        ["lat_cell", "lon_cell"]
    )["frp"]
    .agg(
        historical_mean_frp="mean",
        historical_max_frp="max",
        historical_std_frp="std"
    )
    .reset_index()
)

df = df.merge(
    historical_frp,
    on=["lat_cell", "lon_cell"],
    how="left"
)

df["historical_std_frp"] = (
    df["historical_std_frp"]
    .fillna(0)
)

print(
    df[
        [
            "historical_mean_frp",
            "historical_max_frp",
            "historical_std_frp"
        ]
    ].describe()
)

       historical_mean_frp  historical_max_frp  historical_std_frp
count        702968.000000       702968.000000       702968.000000
mean              8.523616           18.813892            5.626278
std              18.094511           48.170753           17.388804
min               0.000000            0.000000            0.000000
25%               2.535000            3.920000            0.341198
50%               4.400000            7.320000            1.578319
75%               7.500000           14.050000            3.821723
max            2056.400000         2056.400000          869.267579


### Local FRP abnormality

In [6]:
df["local_frp_deviation"] = (
    df["frp"]
    - df["historical_mean_frp"]
)

df["local_frp_zscore"] = (
    df["local_frp_deviation"]
    / df["historical_std_frp"].replace(0, np.nan)
)

df["local_frp_zscore"] = (
    df["local_frp_zscore"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print(
    df["local_frp_zscore"].describe()
)

count    7.029680e+05
mean    -3.881377e-18
std      7.935271e-01
min     -2.885454e+00
25%     -6.362744e-01
50%      0.000000e+00
75%      5.648323e-01
max      1.157371e+01
Name: local_frp_zscore, dtype: float64


### Recent activity

In [7]:
daily_activity = (
    df.groupby(
        [
            "lat_cell",
            "lon_cell",
            "date_only"
        ]
    )
    .size()
    .rename("daily_activity_count")
    .reset_index()
)

df = df.merge(
    daily_activity,
    on=[
        "lat_cell",
        "lon_cell",
        "date_only"
    ],
    how="left"
)

print(
    df["daily_activity_count"]
    .describe()
)

count    702968.000000
mean          3.389503
std           3.218454
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          34.000000
Name: daily_activity_count, dtype: float64


### Regional activity

In [8]:
REGIONAL_CELL_KM = 10.0

REGION_LAT_CELL = REGIONAL_CELL_KM / 111.0
REGION_LON_CELL = REGIONAL_CELL_KM / 100.0

df["region_lat_cell"] = (
    np.floor(
        df["latitude"] / REGION_LAT_CELL
    ).astype("int32")
)

df["region_lon_cell"] = (
    np.floor(
        df["longitude"] / REGION_LON_CELL
    ).astype("int32")
)

regional_activity = (
    df.groupby(
        [
            "region_lat_cell",
            "region_lon_cell",
            "date_only"
        ]
    )
    .size()
    .rename("regional_activity_count")
    .reset_index()
)

df = df.merge(
    regional_activity,
    on=[
        "region_lat_cell",
        "region_lon_cell",
        "date_only"
    ],
    how="left"
)

print(
    df["regional_activity_count"]
    .describe()
)

count    702968.000000
mean         17.177331
std          26.515754
min           1.000000
25%           3.000000
50%           7.000000
75%          20.000000
max         302.000000
Name: regional_activity_count, dtype: float64


### Create contextual feature dataset

In [9]:
context_features = [
    "historical_detection_count",
    "historical_mean_frp",
    "historical_max_frp",
    "historical_std_frp",
    "local_frp_deviation",
    "local_frp_zscore",
    "daily_activity_count",
    "regional_activity_count"
]

context_data = df[
    [
        "latitude",
        "longitude",
        "timestamp",
        "frp"
    ] + context_features
].copy()

print("Context dataset shape:", context_data.shape)

context_data.head()

Context dataset shape: (702968, 12)


,latitude,longitude,timestamp,frp,historical_detection_count,historical_mean_frp,historical_max_frp,historical_std_frp,local_frp_deviation,local_frp_zscore,daily_activity_count,regional_activity_count
0,26.9361,94.0335,2025-01-01 03:50:00,6.5,1,6.50,6.5,0.000000,0.00,0.000000,1,4
1,25.3231,91.6008,2025-01-01 03:51:00,8.0,1,8.00,8.0,0.000000,0.00,0.000000,1,1
2,26.2760,92.8154,2025-01-01 03:51:00,11.6,2,7.49,11.6,5.812418,4.11,0.707107,1,1
3,25.3249,91.5900,2025-01-01 03:51:00,11.3,5,8.54,11.3,3.125348,2.76,0.883102,5,5
4,26.7587,92.7613,2025-01-01 03:51:00,5.3,1,5.30,5.3,0.000000,0.00,0.000000,1,1


### Save it

In [10]:
FEATURE_DIR = (
    PROJECT_ROOT
    / "data"
    / "features"
)

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CONTEXT_FILE = (
    FEATURE_DIR
    / "firms_context_features_prototype.csv"
)

context_data.to_csv(
    CONTEXT_FILE,
    index=False
)

print("Saved:")
print(CONTEXT_FILE)

Saved:
C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\features\firms_context_features_prototype.csv
